# Projet MapReduce avec Spark

## Objectif

Analyser les données NYC Taxi afin de déterminer les heures les plus rentables.

La rentabilité est définie par :

Rentabilité = Total Fare Amount / Total Trip Distance

Nous utiliserons :

- Map : extraction de l'heure
- Reduce : agrégation des montants et distances
- Shuffle : regroupement des clés
- Analyse du Data Skew

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_MapReduce") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark démarré avec succès !")

rdd_test = spark.sparkContext.parallelize([1,2,3,4,5])
print("Count =", rdd_test.count())

rdd_test.saveAsTextFile("hdfs://namenode:9000/test_fichier")
print("Écriture HDFS réussie")

Spark démarré avec succès !
Count = 5
Écriture HDFS réussie


In [3]:
df = spark.read.parquet(
    "hdfs://namenode:9000/tphdfs/data/yellow_tripdata_2010-01.parquet"
)
print("nombre de partition:", df.rdd.getNumPartitions())

print("Nombre de colonnes :", len(df.columns))

print("\nColonnes disponibles :")
print(df.columns)

print("\nAperçu :")
df.show(5)

#print("\nPartitions avant repartition :", df.rdd.getNumPartitions())

# Répartition équilibrée sur 8 partitions
df = df.repartition(8)

print("Partitions après repartition :", df.rdd.getNumPartitions())

nombre de partition: 8
Nombre de colonnes : 18

Colonnes disponibles :
['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'rate_code', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'surcharge', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']

Aperçu :
+---------+-------------------+-------------------+---------------+------------------+------------------+---------------+---------+------------------+------------------+----------------+------------+-----------+---------+-------+----------+------------+------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|  pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag| dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|
+---------+-------------------+-------------------+---------------+-------------

In [3]:
rdd = df.rdd

# print("Nombre de partitions :", rdd.getNumPartitions())

print("Nombre de lignes :", rdd.count())

print("\nPremière ligne :")
print(rdd.first())

print("\nRépartition des données par partition :")

partition_sizes = rdd.mapPartitions(
    lambda it: [sum(1 for _ in it)]
).collect()

for i, size in enumerate(partition_sizes):
    print(f"Partition {i}: {size} lignes")

print("\nTotal :", sum(partition_sizes))

Nombre de lignes : 14863778

Première ligne :
Row(vendor_id='VTS', pickup_datetime='2010-01-13 18:30:00', dropoff_datetime='2010-01-13 18:34:00', passenger_count=4, trip_distance=0.56, pickup_longitude=-73.981798, pickup_latitude=40.7641, rate_code='1', store_and_fwd_flag=None, dropoff_longitude=-73.986723, dropoff_latitude=40.756055, payment_type='CAS', fare_amount=4.1, surcharge=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, total_amount=5.6)

Répartition des données par partition :
Partition 0: 1857972 lignes
Partition 1: 1857972 lignes
Partition 2: 1857972 lignes
Partition 3: 1857972 lignes
Partition 4: 1857972 lignes
Partition 5: 1857972 lignes
Partition 6: 1857973 lignes
Partition 7: 1857973 lignes

Total : 14863778


In [4]:
def is_valid_row(row):
    try:
        return (
            row.fare_amount is not None and
            row.trip_distance is not None and
            float(row.fare_amount) > 0 and
            float(row.trip_distance) > 0
        )
    except:
        return False

clean_rdd = rdd.filter(is_valid_row)

print("Lignes valides :", clean_rdd.count())
print("Lignes supprimées :", rdd.count() - clean_rdd.count())

Lignes valides : 14779891
Lignes supprimées : 83887


# Phase MAP

Transformation :

Map(k,v) → (hour,(fare_amount,trip_distance))

Exemple :

(2010-01-05 08:15:20,12.5,4.2)

↓

(8,(12.5,4.2))

La clé est l'heure de prise en charge.
La valeur contient :

- fare_amount
- trip_distance

In [5]:
from datetime import datetime

def extract_hour_and_values(row):

    try:

        pickup_time = row.pickup_datetime

        if pickup_time is None:
            return None

        if hasattr(pickup_time, "hour"):
            hour = pickup_time.hour

        elif isinstance(pickup_time, str):
            hour = datetime.strptime(
                pickup_time,
                "%Y-%m-%d %H:%M:%S"
            ).hour
        else:
            return None

        return (
            hour,
            (
                float(row.fare_amount),
                float(row.trip_distance)
            )
        )

    except:
        return None

mapped_rdd = clean_rdd \
    .map(extract_hour_and_values) \
    .filter(lambda x: x is not None)

print(mapped_rdd.take(10))

[(18, (4.1, 0.56)), (10, (4.5, 0.6999999999999998)), (11, (3.7, 0.39)), (16, (3.3, 0.4299999999999999)), (17, (7.7, 2.02)), (16, (8.5, 2.4)), (19, (4.9, 0.9)), (16, (4.9, 1.2)), (20, (5.7, 1.4)), (18, (5.3, 1.1))]


# Phase REDUCE

Les valeurs ayant la même clé (heure) sont regroupées.

Reduce(hour)

(fare1,distance1)
+
(fare2,distance2)

↓

(total_fare,total_distance)

In [6]:
def add_tuples(a, b):
    return (
        a[0] + b[0],
        a[1] + b[1]
    )

sum_by_hour = mapped_rdd.reduceByKey(add_tuples)

results = sum_by_hour.collect()

for hour, values in sorted(results):
    print(hour, values)

0 (5768537.849999974, 1745146.809999998)
1 (4327068.68999999, 1322300.2500000014)
2 (3419346.430000007, 1061390.8600000022)
3 (2665702.8300000024, 846734.459999998)
4 (2068632.0800000047, 685693.3700000017)
5 (1572819.9900000042, 536566.3199999991)
6 (2732634.880000006, 882790.4200000004)
7 (4708802.229999969, 1372313.7799999968)
8 (6135321.479999915, 1634310.0199999965)
9 (6019069.429999882, 1579472.1400000025)
10 (5549826.05999992, 1486125.8199999982)
11 (5850196.579999922, 1570444.939999999)
12 (6435928.609999874, 1705619.630000003)
13 (6636589.399999883, 1775534.840000005)
14 (7069747.439999877, 1908309.2599999998)
15 (7109516.019999909, 1950868.4499999995)
16 (6396147.209999886, 1778323.8099999998)
17 (7383330.109999902, 1996284.8099999907)
18 (8481118.309999838, 2250869.0200000037)
19 (8494832.569999866, 2299303.8999999873)
20 (7788402.969999897, 2219977.449999995)
21 (7539661.559999913, 2218290.7799999877)
22 (7419143.079999905, 2201203.600000005)
23 (6802550.979999943, 2038099.

In [7]:
print("Partitioner utilisé :")
print(sum_by_hour.partitioner)

Partitioner utilisé :


# HashPartitioner

Spark distribue les clés selon :

partition = hash(key) mod R

où :

- key = heure
- R = nombre de reducers

Toutes les données d'une même heure sont envoyées au même reducer.

In [8]:
def calculate_profitability(x):

    hour, (fare, distance) = x

    if distance == 0:
        return (hour, 0)

    return (
        hour,
        round(fare / distance, 2)
    )

profitability_by_hour = sum_by_hour.map(
    calculate_profitability
)

results = profitability_by_hour \
    .sortByKey() \
    .collect()

for hour, profit in results:
    print(
        f"Heure {hour}:00 -> {profit} $/km"
    )

Heure 0:00 -> 3.31 $/km
Heure 1:00 -> 3.27 $/km
Heure 2:00 -> 3.22 $/km
Heure 3:00 -> 3.15 $/km
Heure 4:00 -> 3.02 $/km
Heure 5:00 -> 2.93 $/km
Heure 6:00 -> 3.1 $/km
Heure 7:00 -> 3.43 $/km
Heure 8:00 -> 3.75 $/km
Heure 9:00 -> 3.81 $/km
Heure 10:00 -> 3.73 $/km
Heure 11:00 -> 3.73 $/km
Heure 12:00 -> 3.77 $/km
Heure 13:00 -> 3.74 $/km
Heure 14:00 -> 3.7 $/km
Heure 15:00 -> 3.64 $/km
Heure 16:00 -> 3.6 $/km
Heure 17:00 -> 3.7 $/km
Heure 18:00 -> 3.77 $/km
Heure 19:00 -> 3.69 $/km
Heure 20:00 -> 3.51 $/km
Heure 21:00 -> 3.4 $/km
Heure 22:00 -> 3.37 $/km
Heure 23:00 -> 3.34 $/km


In [9]:
output_path = \
"hdfs://namenode:9000/tphdfs/output/profitability_by_hour"

output_rdd = profitability_by_hour.map(
    lambda x: f"{x[0]},{x[1]}"
)

try:

    conf = spark.sparkContext._jsc.hadoopConfiguration()

    fs = spark.sparkContext._jvm \
        .org.apache.hadoop.fs.FileSystem \
        .get(conf)

    path = spark.sparkContext._jvm \
        .org.apache.hadoop.fs.Path(output_path)

    if fs.exists(path):
        fs.delete(path, True)

except:
    pass

output_rdd.saveAsTextFile(output_path)

print("Résultats sauvegardés")

Résultats sauvegardés


In [10]:
print("=== ANALYSE DU SHUFFLE ===")

print(
    "Partitions avant reduceByKey :",
    mapped_rdd.getNumPartitions()
)

def get_partition_sizes(rdd):
    return rdd.mapPartitions(
        lambda it: [sum(1 for _ in it)]
    ).collect()

partition_sizes = get_partition_sizes(mapped_rdd)

print("Taille partitions :")
print(partition_sizes)

print(
    "Nombre total d'enregistrements :",
    sum(partition_sizes)
)

=== ANALYSE DU SHUFFLE ===
Partitions avant reduceByKey : 8
Taille partitions :
[1847395, 1847620, 1847308, 1847656, 1847265, 1847469, 1847500, 1847678]
Nombre total d'enregistrements : 14779891


In [11]:
print("=== VOLUME DU SHUFFLE ===")

nb_records = mapped_rdd.count()

print("Nombre d'enregistrements :", nb_records)

estimated_volume = nb_records * 24

print(
    "Volume estimé du shuffle :",
    estimated_volume,
    "octets"
)

=== VOLUME DU SHUFFLE ===
Nombre d'enregistrements : 14779891
Volume estimé du shuffle : 354717384 octets


# Volume du Shuffle

Formule :

V ≈ N × S

où :

- N = nombre d'enregistrements
- S = taille moyenne d'un tuple

Le volume du shuffle augmente avec la quantité de données.

In [12]:
print("=== IMPACT DU NOMBRE DE REDUCERS ===")

for r in [1,2,4,8]:

    reduced = mapped_rdd.reduceByKey(
        add_tuples,
        numPartitions=r
    )

    print(
        f"Reducers={r} "
        f"-> partitions={reduced.getNumPartitions()}"
    )

=== IMPACT DU NOMBRE DE REDUCERS ===
Reducers=1 -> partitions=1
Reducers=2 -> partitions=2
Reducers=4 -> partitions=4
Reducers=8 -> partitions=8


# Impact du nombre de Reducers

Peu de reducers :

- moins de parallélisme

Beaucoup de reducers :

- plus de parallélisme
- plus de coût de coordination

Une valeur intermédiaire offre généralement les meilleures performances.

In [13]:
from random import random

def create_skewed_data(row):

    if random() < 0.8:

        return (
            8,
            (
                float(row.fare_amount),
                float(row.trip_distance)
            )
        )

    return (
        int(random()*24),
        (
            float(row.fare_amount),
            float(row.trip_distance)
        )
    )

sample_for_skew = clean_rdd.sample(False,0.1,42)

skewed_rdd = sample_for_skew.map(
    create_skewed_data
)

distribution = skewed_rdd.countByKey()

print(distribution)

defaultdict(<class 'int'>, {8: 1194870, 14: 12309, 6: 12417, 0: 12523, 1: 12114, 10: 12931, 21: 12827, 22: 12135, 19: 12557, 18: 12582, 17: 12571, 3: 12193, 15: 12518, 4: 12220, 7: 12024, 20: 11521, 5: 12110, 23: 12212, 2: 11952, 11: 11856, 16: 12175, 13: 12114, 12: 11802, 9: 12208})


In [14]:
print("=== IMPACT DU DATA SKEW ===")

for h, c in sorted(distribution.items()):
    print(f"Heure {h} : {c}")

print("\nT(job) ≈ T(reducer le plus lent)")

=== IMPACT DU DATA SKEW ===
Heure 0 : 12523
Heure 1 : 12114
Heure 2 : 11952
Heure 3 : 12193
Heure 4 : 12220
Heure 5 : 12110
Heure 6 : 12417
Heure 7 : 12024
Heure 8 : 1194870
Heure 9 : 12208
Heure 10 : 12931
Heure 11 : 11856
Heure 12 : 11802
Heure 13 : 12114
Heure 14 : 12309
Heure 15 : 12518
Heure 16 : 12175
Heure 17 : 12571
Heure 18 : 12582
Heure 19 : 12557
Heure 20 : 11521
Heure 21 : 12827
Heure 22 : 12135
Heure 23 : 12212

T(job) ≈ T(reducer le plus lent)


# Conclusion

Cette implémentation MapReduce permet d'analyser la rentabilité des taxis selon l'heure.

Le shuffle est nécessaire pour regrouper les données partageant la même clé.

L'étude du Data Skew montre qu'une mauvaise répartition des clés peut ralentir fortement l'exécution.

MapReduce demeure un modèle fondamental pour comprendre le traitement distribué de grandes masses de données.

In [15]:
print("=== FICHIERS HDFS ===")

files = spark.sparkContext._jsc.hadoopConfiguration()

fs = spark.sparkContext._jvm \
    .org.apache.hadoop.fs.FileSystem \
    .get(files)

path = spark.sparkContext._jvm \
    .org.apache.hadoop.fs.Path(
        "/tphdfs/output/profitability_by_hour"
    )

if fs.exists(path):

    for f in fs.listStatus(path):

        print(
            f"{f.getPath().getName()} "
            f"- {f.getLen()} bytes"
        )

=== FICHIERS HDFS ===
_SUCCESS - 0 bytes
part-00000 - 21 bytes
part-00001 - 21 bytes
part-00002 - 23 bytes
part-00003 - 23 bytes
part-00004 - 23 bytes
part-00005 - 22 bytes
part-00006 - 21 bytes
part-00007 - 23 bytes
